In [22]:
import os
import re

import mlflow
import numpy as np
import polars as pl
import plotly.colors as pc
import plotly.graph_objects as go
from pymoo.indicators.hv import HV

from paretodo.dashboard.utils.types_and_constants import LOCAL_TEMP_DIR

from setup.plotly_theme import register_theme

register_theme()

PDF_FOLDER = ".local/pdf"
os.makedirs(PDF_FOLDER, exist_ok=True)

# same colours / symbols as paper_benchmarks
name_dict = {
    "pareto_expert": "Pareto Expert",
    "pareto_expert_80": "Pareto Expert 80%",
    "pareto_expert_95": "Pareto Expert 95%",
}
color_dict = {
    "pareto_expert": "#0072B2",
    "pareto_expert_80": "#56B4E9",
    "pareto_expert_95": "#CCBB44",
}
symbol_dict = {
    "pareto_expert": "circle",
    "pareto_expert_80": "square",
    "pareto_expert_95": "diamond",
}
MARKER_SIZE = 14
LEGEND_MARKER_SIZE = 20

In [23]:
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "Rev-Benchmark_Opt-15-09-2026"
MLFLOW_EXPERIMENT_CONSTRAINED = "Rev-Benchmark_OptC-15-09-2026"

MU = "obj:max(μ(product_concentration))"
SIGMA = "obj:max(σ(product_concentration))"
P_PURITY = "obj:max(P(purity))"
# both objectives are maximised and non-negative, so the origin is a valid reference point
REFERENCE_POINT = np.array([0.0, 0.0])
# purity constraint of the constrained problem (bio_mean_std_cpurity.json), used to decide which experiments are feasible
PURITY_CONSTRAINT = 0.8

client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
# pymoo minimises, so negate the maximised objectives (and the reference point)
hv_indicator = HV(ref_point=-REFERENCE_POINT)


def get_expert_runs(experiment_name):
    experiment = client.get_experiment_by_name(experiment_name)
    runs = client.search_runs(experiment_ids=[experiment.experiment_id])
    return [r for r in runs if r.data.params.get("sampling_mode") == "pareto_expert"]


def purity_threshold(run):
    """Lower bound on P(purity) the expert used to discard Pareto points (None if unconstrained)."""
    match = re.search(r"\(([\d.]+),", run.data.params.get("discard_values", ""))
    return float(match.group(1)) if match else None


def run_trajectory(run, threshold=None):
    """Hypervolume of the predicted Pareto front and best experiment so far, per iteration."""
    run_id = run.info.run_id
    temp_dir = os.path.join(LOCAL_TEMP_DIR, run_id)
    os.makedirs(temp_dir, exist_ok=True)
    iterations = sorted(a.path for a in client.list_artifacts(run_id) if a.path.startswith("Iteration_"))

    rows = []
    for iteration in iterations:
        pareto_df = pl.read_csv(client.download_artifacts(run_id=run_id, path=f"{iteration}/pareto_front_predictions.csv", dst_path=temp_dir))
        experiments_df = pl.read_csv(client.download_artifacts(run_id=run_id, path=f"{iteration}/experiments.csv", dst_path=temp_dir))
        # experiments.csv holds the experiments the model of that iteration was trained on
        n_experiments = len(experiments_df)
        if threshold is not None:
            # only the part of the front the expert could select from, and only feasible experiments
            pareto_df = pareto_df.filter(pl.col(P_PURITY) >= threshold)
            experiments_df = experiments_df.filter(pl.col("purity") >= PURITY_CONSTRAINT)
        points = pareto_df.select([MU, SIGMA]).to_numpy()
        best = experiments_df["product_concentration"].max()
        rows.append({
            "Iteration": int(iteration.split("_")[1]),
            "Experiments": n_experiments,
            "Hypervolume": hv_indicator(-points) if len(points) else 0.0,
            # None (no feasible experiment yet) is skipped when averaging over runs
            "Best Product Concentration": best,
        })
    return pl.DataFrame(rows).with_columns(pl.lit(run.info.run_id).alias("run_id"))


def mean_std(trajectories):
    return (
        pl.concat(trajectories)
        .group_by("Experiments")
        .agg(
            pl.len().alias("n_runs"),
            pl.col("Hypervolume").mean().alias("Hypervolume_mean"),
            pl.col("Hypervolume").std().alias("Hypervolume_std"),
            pl.col("Best Product Concentration").mean().alias("Best Product Concentration_mean"),
            pl.col("Best Product Concentration").std().alias("Best Product Concentration_std"),
        )
        .sort("Experiments")
    )

In [24]:
# unconstrained case: all Pareto Expert runs
expert_runs = get_expert_runs(MLFLOW_EXPERIMENT)
print(f"{len(expert_runs)} Pareto Expert runs")
summary = {"pareto_expert": mean_std([run_trajectory(r) for r in expert_runs])}

# constrained case: Pareto Expert runs grouped by the purity threshold used for discarding
constrained_runs = get_expert_runs(MLFLOW_EXPERIMENT_CONSTRAINED)
constrained_summary = {}
for threshold in sorted({purity_threshold(r) for r in constrained_runs}):
    runs = [r for r in constrained_runs if purity_threshold(r) == threshold]
    print(f"{len(runs)} constrained Pareto Expert runs with P(purity) >= {threshold}")
    constrained_summary[f"pareto_expert_{round(threshold * 100)}"] = mean_std([run_trajectory(r, threshold) for r in runs])

2026-09-24 18:14:42,452 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-09-24 18:14:42,453 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


5 Pareto Expert runs


2026-09-24 18:14:43,809 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-09-24 18:14:43,810 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


5 constrained Pareto Expert runs with P(purity) >= 0.8


5 constrained Pareto Expert runs with P(purity) >= 0.95


In [30]:
def _hex_to_rgba(hex_color, alpha):
    r, g, b = pc.hex_to_rgb(hex_color)
    return f"rgba({r},{g},{b},{alpha})"


def plot_mean_std(summary, column, title, y_title, filename, **yaxis_kwargs):
    fig = go.Figure()
    for algo, df in summary.items():
        color = color_dict[algo]
        mean = df[f"{column}_mean"]
        std = df[f"{column}_std"]
        fig.add_trace(go.Scatter(x=df["Experiments"], y=mean + std, line=dict(width=0), showlegend=False, hoverinfo="skip", legendgroup=algo))
        fig.add_trace(
            go.Scatter(
                x=df["Experiments"],
                y=mean - std,
                line=dict(width=0),
                fill="tonexty",
                fillcolor=_hex_to_rgba(color, 0.2),
                showlegend=False,
                hoverinfo="skip",
                legendgroup=algo,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=df["Experiments"],
                y=mean,
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=2, color=color),
                marker=dict(size=MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                showlegend=False,
                legendgroup=algo,
            )
        )
        # legend-only trace so the legend line is thicker than the plotted line
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                name=name_dict[algo],
                mode="lines+markers",
                line=dict(width=6, color=color),
                marker=dict(size=LEGEND_MARKER_SIZE, color=color, symbol=symbol_dict[algo]),
                legendgroup=algo,
            )
        )
    fig.update_layout(legend=dict(bgcolor="rgba(255,255,255,0.5)", groupclick="togglegroup", font=dict(size=30), x=0.02, y=1), margin=dict(l=120, r=20, t=100, b=120), width=1200, height=800, title=title, title_font_size=30)
    fig.update_xaxes(title_text="Number of Experiments", title_font_size=30, tickfont=dict(size=30))
    fig.update_yaxes(title_text=y_title, title_font_size=30, tickfont=dict(size=30), title_standoff=20, **yaxis_kwargs)
    fig.show()
    fig.update_layout(paper_bgcolor="rgba(0,0,0,0)")
    fig.write_image(f"{PDF_FOLDER}/{filename}.pdf")
    return fig

In [31]:
fig = plot_mean_std(
    summary,
    "Hypervolume",
    "Mean and One Standard Deviation of Pareto Front Hypervolume",
    "Hypervolume [a.u.]",
    "hypervolume_pareto_expert",
    showticklabels=False,
    ticks="",
)

2026-09-24 18:15:17,412 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-09-24 18:15:17,413 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-09-24 18:15:17,419 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-09-24 18:15:17,420 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-24 18:15:17,421 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmplgeyvrkn.
2026-09-24 18:15:17,422 - choreographer.browser_async - INFO - Opening browser.
2026-09-24 18:15:17,424 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpggnv6x7f.
2026-09-24 18:15:17,425 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpggnv6x7f
2026-09-24 18:15:17,872 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmplgeyvrkn/index.html
2026-09-24 18:15:18,457 - kaleido.kaleido - INFO - Getting tab from queue 

In [32]:
fig = plot_mean_std(
    summary,
    "Best Product Concentration",
    "Mean and One Standard Deviation of Best Product Concentration",
    "Best Product Concentration",
    "best_objective_pareto_expert",
)

2026-09-24 18:15:18,768 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-09-24 18:15:18,768 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-09-24 18:15:18,773 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-09-24 18:15:18,774 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-24 18:15:18,774 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpd0nosve_.
2026-09-24 18:15:18,775 - choreographer.browser_async - INFO - Opening browser.
2026-09-24 18:15:18,776 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpu5krcn0c.
2026-09-24 18:15:18,776 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpu5krcn0c
2026-09-24 18:15:19,200 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpd0nosve_/index.html
2026-09-24 18:15:19,793 - kaleido.kaleido - INFO - Getting tab from queue 

In [33]:
fig = plot_mean_std(
    constrained_summary,
    "Hypervolume",
    "Mean and One Standard Deviation of Feasible Pareto Front Hypervolume",
    "Hypervolume [a.u.]",
    "hypervolume_pareto_expert_constrained",
    showticklabels=False,
    ticks="",
)

2026-09-24 18:15:20,117 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-09-24 18:15:20,118 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-09-24 18:15:20,123 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-09-24 18:15:20,124 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-24 18:15:20,125 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp5jfqy6nz.
2026-09-24 18:15:20,126 - choreographer.browser_async - INFO - Opening browser.
2026-09-24 18:15:20,127 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpjnydio7_.
2026-09-24 18:15:20,127 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpjnydio7_
2026-09-24 18:15:20,601 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmp5jfqy6nz/index.html
2026-09-24 18:15:21,169 - kaleido.kaleido - INFO - Getting tab from queue 

In [34]:
fig = plot_mean_std(
    constrained_summary,
    "Best Product Concentration",
    "Mean and One Standard Deviation of Best Feasible Product Concentration",
    "Best Product Concentration",
    "best_objective_pareto_expert_constrained",
)

2026-09-24 18:15:21,424 - choreographer.utils._tmpfile - INFO - TemporaryDirectory.cleanup() worked.
2026-09-24 18:15:21,425 - choreographer.utils._tmpfile - INFO - shutil.rmtree worked.


2026-09-24 18:15:21,430 - choreographer.browsers.chromium - INFO - Chromium init'ed with kwargs {}
2026-09-24 18:15:21,431 - choreographer.browsers.chromium - INFO - Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
2026-09-24 18:15:21,432 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvy3g7n9y.
2026-09-24 18:15:21,432 - choreographer.browser_async - INFO - Opening browser.
2026-09-24 18:15:21,433 - choreographer.utils._tmpfile - INFO - Temp directory created: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpne4ril4l.
2026-09-24 18:15:21,433 - choreographer.browsers.chromium - INFO - Temporary directory at: /var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpne4ril4l
2026-09-24 18:15:21,866 - kaleido.kaleido - INFO - Conforming 1 to file:///var/folders/tq/2k8k0v4s3lv73ssw_kqwr2h00000gp/T/tmpvy3g7n9y/index.html
2026-09-24 18:15:22,418 - kaleido.kaleido - INFO - Getting tab from queue 